# My task write-up: Implementing the production navigation inference endpoint

**Task:** Add the production-facing ML inference path that serves our approved WalkBuddy navigation model through the existing backend. Scope was *only* ML model serving and its API contract — no auth, routing, chat, OCR, or speech.

**What I built (in my own words):** I added a brand-new endpoint, `POST /ml/navigate`, that takes a camera frame, runs it through the YOLO model we already load at startup, and returns the detections plus a spoken-style guidance message in a clean, documented JSON shape. On top of that I added a **mock mode** (an env flag) so my teammates on the frontend can build against the API *before* a real navigation model exists.

> **Note (updated after review):** I originally also shipped my own `GET /ml/model-info`, but during review I removed it because the newly-merged `ml_runtime` package already provides the authoritative one. See the "Review feedback and rebase" section below — the rest of this write-up reflects the **final, post-review** code.

The most important rule I gave myself was: **this is a pure addition**. I did not touch `/vision`, `/ws/vision`, or any other existing route. I reused the model, the adapter, and the helper functions that were already there instead of writing new inference logic.

Files I added or changed (final state):

- `routers/ml_inference.py` — **new**, one endpoint (`POST /ml/navigate`) + mock mode, wired into the shared ML runtime.
- `main.py` — **changed**, 2 lines to register my router (unchanged this round; `ml_runtime`'s router was already registered by the merged infra).
- `tests/test_ml_inference.py` — **new**, router-isolated tests (no real weights).
- `tests/test_ml_inference_integration.py` — **new**, real-app integration tests (added in review).
- `requirements-dev.txt` — **new**, so the tests run in CI.
- `docs/ml_inference.md` — **new**, human-readable docs for the endpoint.

## 0. Review feedback and rebase (what changed after I opened the PR)

After I finished the first version I opened **PR #179** and the team lead reviewed it. In the time between me starting the work and the review, a *lot* of new ML infrastructure had been merged into `t2-2026-development` — two packages I hadn't seen when I wrote my code:

- **`ml_contract/`** — a central, approved definition of the navigation taxonomy (the 8 MVP classes: `person, stairs, door, chair, table, pole, bicycle, vehicle`) and each class's base severity/priority. It's meant to be the single source of truth.
- **`ml_runtime/`** — shared runtime plumbing: an authoritative `GET /ml/model-info` (plus `/ml/health`, `/ml/ready`, `/ml/metrics`), an `InferenceMetrics` object for tracking attempts/successes/failures/latency, and a stable error format for ML failures.

Because my branch was based on an older commit, the first thing I did was **rebase my branch onto the latest `t2-2026-development`** so I was building on top of that new infrastructure instead of colliding with it. Then I worked through the **5 integration fixes** the team lead asked for:

1. **Removed my own `GET /ml/model-info`** — `ml_runtime` already owns the authoritative one, so I keep only `POST /ml/navigate`.
2. **Mock taxonomy now derived from `ml_contract`** (the approved 8 nav classes) instead of a hardcoded copy.
3. **Mock priority now comes from the contract's base severity** (so `table` is `MEDIUM`, not `HIGH`).
4. **`/ml/navigate` now plugs into the shared ML-runtime metrics + stable error format**, exactly like `/vision`, instead of my own 503/500.
5. **Added a real-app integration test** on top of my router-isolated ones.

The rest of this notebook has been updated to show the **final, post-review code**. Where it matters, I call out what changed and why.

## 1. What I found in the existing backend before writing any code

Before touching anything, I traced how vision already works so I could reuse it instead of reinventing it. Here's what I found:

- **The model is loaded once at startup.** In `main.py` there's a `lifespan` function that runs when the server boots. It does `app.state.yolo = YOLO(str(YOLO_MODEL_PATH))`. So the model lives on `app.state.yolo` and every request can grab it from there — I don't need to load it myself. The path comes from an env var `WALKBUDDY_MODEL_DIR` (for Docker) or falls back to `ML_side/models/best.pt`. If loading fails it sets `app.state.yolo = None` instead of crashing.
- **There's a concurrency limiter.** `app.state.vision_limiter` is an `anyio.CapacityLimiter(1)`, which means only one YOLO inference runs at a time. The existing `/vision` endpoint uses it, so I use it too to stay consistent.
- **`/vision` and `/ws/vision` both call the same `vision_adapter`.** The real detection logic lives in `adapters/vision_adapter.py`. It runs `model.predict(...)` and returns a dict. I found the exact shape it returns: an `image_id`, a list of `detections` (each with `category`, `confidence`, `bbox`, `direction`, `priority`), and `metadata` with the image shape. The class *name* for each detection comes straight from the model itself (`result.names[cls_id]`) — it's **not** hardcoded anywhere.
- **Guidance messages are built by helpers.** `routers/ai_service.py` has `_event_from_detection(...)` and `_guidance_payload(...)`. The second one first asks the safety gate if anything dangerous is ahead, and otherwise turns detections into a short message like "table ahead". I decided to reuse both so my endpoint gives the same guidance as the existing ones.

**The one thing that surprised me (the 7-vs-8 class discrepancy):** the training config `ML_side/config/newdata.yaml` lists **8** classes (it adds `couch`), but the docs for the actual `best.pt` file (`ML_side/models/README.md`) say the verified weights only have **7** classes. So the config and the real model file don't match. I decided **not** to hardcode either list — instead I read the class names from the loaded model at runtime, so my code is correct no matter which weights are loaded. Fixing the lineage mismatch is someone else's follow-up job.

The code cell below is the existing code I studied (copied from `main.py` and `vision_adapter.py`) — this is what I built on top of, not code I wrote.

In [ ]:
# --- EXISTING CODE I STUDIED (not mine) ---

# main.py — the model is loaded once at startup and stored on app.state
#     try:
#         logger.info(f"Loading YOLO from {YOLO_MODEL_PATH}")
#         app.state.yolo = YOLO(str(YOLO_MODEL_PATH))
#         logger.info("\u2705 YOLO ready")
#     except Exception as e:
#         logger.error(f"\u274c YOLO load failed: {e}")
#         app.state.yolo = None
#     ...
#     app.state.vision_limiter = anyio.CapacityLimiter(1)   # one inference at a time


# adapters/vision_adapter.py — the detection logic + the dict shape it returns
def vision_adapter(model, image_path):
    results = model.predict(source=image_path, conf=0.25, iou=0.45, verbose=False)
    result = results[0]
    detections = []
    # ... reads image size, then for each box: ...
    #     label = result.names[cls_id]        # <-- class NAME comes from the model itself
    #     direction = calculate_spatial_position(bbox, image_width)  # left / right / ahead
    #     priority = get_priority(label)      # HIGH / MEDIUM / LOW
    #     detections.append({
    #         "category": label,
    #         "confidence": round(conf, 3),
    #         "bbox": {"x_min": ..., "y_min": ..., "x_max": ..., "y_max": ...},
    #         "direction": direction,
    #         "priority": priority,
    #     })
    return {
        "image_id": "<file stem>",
        "detections": detections,
        "metadata": {"image_shape": [image_height, image_width]},
    }


# routers/ai_service.py — the helpers I reused for guidance + memory
def _event_from_detection(detection):
    return {
        "label": detection["category"],
        "direction": detection.get("direction", "ahead"),
        "distance_m": None,
        "confidence": detection["confidence"],
        # ... plus motion fields (track_id, is_moving, approaching, ...)
    }

def _guidance_payload(result, max_messages=1):
    # 1) ask the safety gate for a STOP message if a hazard is ahead
    # 2) otherwise build a short spoken message from the detections
    # returns (message_string, risk_level_string)
    ...


## 2. File I added: `routers/ml_inference.py` (post-review)

This is the main file — the actual endpoint. Below is the **current** version after the 5 review fixes. Here's how it works and why:

**Why a new file?** I went with a new file `routers/ml_inference.py` because the task is specifically about *production ML serving*, and keeping it separate makes the scope obvious and easy to review. It uses `APIRouter(prefix="/ml")`.

**Only one endpoint now (`POST /ml/navigate`).** I *removed* my own `GET /ml/model-info` in review. The merged `ml_runtime` package already registers an authoritative `/ml/model-info`, so mine was a second handler on the exact same path. That's a bad idea: which one answers depends on router registration order, so it's confusing and fragile, and it means two different "sources of truth" for the class list. Deleting mine leaves exactly one, owned by the team's shared runtime.

**How `POST /ml/navigate` works:**
1. If mock mode is on, it returns a fixed fake result and stops (more below).
2. It grabs the already-loaded model from `request.app.state.yolo`. If that's falsy, it returns **503** using the shared error format.
3. It reads the class list from the model with `_model_classes()` (reads `yolo.names`, so it's never hardcoded).
4. If the uploaded file is empty, it short-circuits with an empty-but-valid response.
5. Otherwise it writes the frame to a temp file, waits for the shared `vision_limiter` slot, and runs `vision_adapter` in a worker thread — wrapped in the shared metrics helpers.
6. It saves each detection into `state.memory` and builds the guidance message with `_guidance_payload`.
7. It returns the tidy JSON contract: `model`, `classes`, `detections`, `guidance_message`, `risk_level`, `inference_time_ms`, `image_id`.
8. A `finally` block always deletes the temp file.

**Shared metrics + stable errors (review fix #4).** Instead of my own `HTTPException(503/500)`, I now do exactly what `/vision` does. I call `_begin_vision_metrics()` before the model runs and `_finish_vision_metrics(..., successful=True/False)` after, so attempts / successes / failures / latency all show up in the shared `ml_runtime` metrics (visible at `GET /ml/metrics`). And I return the **stable error payloads** from `ml_runtime.errors`: `model_unavailable_error()` (503) and `inference_failed_error()` (500). So every ML endpoint reports failures the same way.

**Mock taxonomy comes from the contract now (review fixes #2 and #3).** The mock used to report the old indoor classes with `table` as `HIGH`. Now the mock taxonomy is the approved navigation MVP classes — `person, stairs, door, chair, table, pole, bicycle, vehicle` — and I **derive** it from the shared contract instead of typing a second copy:

```python
MOCK_CLASSES = [nav_class.name for nav_class in NAVIGATION_CLASSES]
```

Likewise the mock detection's `priority` is read from the contract's base severity, so it stays in sync automatically. In the contract `table` is `MEDIUM` (not `HIGH`), so:

```python
"priority": get_base_severity("table").name   # -> "MEDIUM"
```

**Mock mode (`WALKBUDDY_ML_MOCK`):** still an env flag, off by default, and it still works with no weights loaded (`app.state.yolo` can be `None`). The `model` field becomes `walkbuddy-yolo-mock` so the frontend can tell it's fake data.

The full current file is below.

In [ ]:
"""
Production ML inference endpoint for the WalkBuddy navigation model.

This router is a *pure addition* on top of the existing vision pipeline. It
reuses the already-loaded YOLO model (`app.state.yolo`), the shared capacity
limiter (`app.state.vision_limiter`), the `vision_adapter` detection logic, the
`_event_from_detection` / `_guidance_payload` helpers, the shared ML-runtime
metrics + error helpers, and `state.memory`. It introduces no new inference
logic and does not touch `/vision`, `/ws/vision`, or any other route.

It exposes exactly one endpoint: `POST /ml/navigate`. The authoritative
`GET /ml/model-info` (and `/ml/health`, `/ml/ready`, `/ml/metrics`) are owned by
`ml_runtime.router`; this router deliberately does NOT define a second
`/ml/model-info` handler.

Real path: `model`/`classes` are read from `app.state.yolo.names` at request
time, so the contract works unchanged for whatever weights are loaded, without
hardcoding a taxonomy.

Mock mode: set `WALKBUDDY_ML_MOCK=1` (default off) to make `/ml/navigate` return
a deterministic fake result in the same contract, with no weights and no
inference. The mock taxonomy and per-class priority are DERIVED from
`ml_contract.navigation_semantics` (the approved MVP navigation classes), not
hardcoded here. This lets the frontend / API be developed before a real
navigation model exists.
"""

import os
import time
import tempfile
import logging

import anyio
from fastapi import APIRouter, UploadFile, File, Request
from fastapi.responses import JSONResponse

from adapters.vision_adapter import vision_adapter
from internal import state
from routers.ai_service import (
    _event_from_detection,
    _guidance_payload,
    _begin_vision_metrics,
    _finish_vision_metrics,
)
from ml_contract import NAVIGATION_CLASSES, get_base_severity
from ml_runtime import inference_failed_error, model_unavailable_error

logger = logging.getLogger(__name__)
router = APIRouter(prefix="/ml", tags=["ml"])

# ── Mock mode ───────────────────────────────────────────────────────────────
# When WALKBUDDY_ML_MOCK is truthy the endpoint returns a deterministic fake
# result (no weights, no inference) so the API can be developed before a real
# navigation model exists. Off by default.
_TRUTHY = {"1", "true", "yes", "on"}

# Approved navigation MVP taxonomy, derived from the shared contract so there is
# a single source of truth (never a second hardcoded copy).
MOCK_CLASSES = [nav_class.name for nav_class in NAVIGATION_CLASSES]
MOCK_MODEL = "walkbuddy-yolo-mock"

# One deterministic detection in the exact shape vision_adapter produces. Its
# priority is read from the contract's base severity for that class (e.g. the
# contract rates "table" as MEDIUM), not hardcoded.
_MOCK_CATEGORY = "table"
_MOCK_RESULT = {
    "image_id": "mock",
    "detections": [
        {
            "category": _MOCK_CATEGORY,
            "confidence": 0.87,
            "bbox": {"x_min": 220, "y_min": 180, "x_max": 420, "y_max": 400},
            "direction": "ahead",
            "priority": get_base_severity(_MOCK_CATEGORY).name,
        }
    ],
    "metadata": {"image_shape": [480, 640]},
}


def _mock_enabled() -> bool:
    return os.getenv("WALKBUDDY_ML_MOCK", "").strip().lower() in _TRUTHY


def _model_classes(yolo) -> list[str]:
    """Return the model's class names in class-index order.

    Reads directly from the loaded model's `.names` so the contract reflects the
    actual weights rather than a hardcoded list. Ultralytics exposes `.names` as
    a dict keyed by int class id; a plain list is also handled defensively.
    """
    names = getattr(yolo, "names", None)
    if names is None:
        return []
    if isinstance(names, dict):
        return [names[key] for key in sorted(names)]
    return list(names)


def _model_descriptor(classes: list[str]) -> str:
    return f"walkbuddy-yolo-{len(classes)}class"


@router.post("/navigate")
async def navigate_endpoint(request: Request, file: UploadFile = File(...)):
    """Run the approved navigation model on a single frame.

    Contract (200):
        {
          "model": str,                 # derived from app.state.yolo.names
          "classes": list[str],         # class names in index order
          "detections": list[dict],     # exactly what vision_adapter returns
          "guidance_message": str,
          "risk_level": str,
          "inference_time_ms": int,
          "image_id": str | None,
        }

    Errors use the stable ML error format from `ml_runtime.errors`:
      503 -> model_unavailable_error(); 500 -> inference_failed_error().
    Successful and failed inferences are recorded in the shared ML-runtime
    metrics, exactly like `/vision`.
    """
    if _mock_enabled():
        result = _MOCK_RESULT
        for d in result["detections"]:
            state.memory.add_event(**_event_from_detection(d))
        guidance, risk_level = _guidance_payload(result, max_messages=3)
        return {
            "model": MOCK_MODEL,
            "classes": list(MOCK_CLASSES),
            "detections": result["detections"],
            "guidance_message": guidance,
            "risk_level": risk_level,
            "inference_time_ms": 0,
            "image_id": result["image_id"],
        }

    if not request.app.state.yolo:
        return JSONResponse(status_code=503, content=model_unavailable_error())

    classes = _model_classes(request.app.state.yolo)

    content = await file.read()
    if not content:
        return {
            "model": _model_descriptor(classes),
            "classes": classes,
            "detections": [],
            "guidance_message": "",
            "risk_level": "CLEAR",
            "inference_time_ms": 0,
            "image_id": None,
        }

    temp_path = None
    try:
        suffix = os.path.splitext(file.filename or "frame.jpg")[1] or ".jpg"
        with tempfile.NamedTemporaryFile(delete=False, suffix=suffix) as f:
            f.write(content)
            temp_path = f.name

        try:
            async with request.app.state.vision_limiter:
                t0 = time.monotonic()
                metrics_started_at = _begin_vision_metrics(request.app)
                try:
                    result = await anyio.to_thread.run_sync(
                        vision_adapter,
                        request.app.state.yolo,
                        temp_path,
                    )
                except Exception:
                    _finish_vision_metrics(
                        request.app, metrics_started_at, successful=False
                    )
                    raise
                _finish_vision_metrics(request.app, metrics_started_at, successful=True)
                inference_ms = int((time.monotonic() - t0) * 1000)
        except Exception:
            logger.exception("ML navigate adapter error")
            return JSONResponse(status_code=500, content=inference_failed_error())

        for d in result["detections"]:
            state.memory.add_event(**_event_from_detection(d))

        guidance, risk_level = _guidance_payload(result, max_messages=3)

        return {
            "model": _model_descriptor(classes),
            "classes": classes,
            "detections": result["detections"],
            "guidance_message": guidance,
            "risk_level": risk_level,
            "inference_time_ms": inference_ms,
            "image_id": result["image_id"],
        }

    finally:
        if temp_path and os.path.exists(temp_path):
            os.unlink(temp_path)


## 3. File I changed: `main.py` (only 2 lines)

To keep this a pure addition, I only added **two lines** to `main.py`: one to import my router, and one to register it on the app. I put them right next to where the existing routers are imported and included, so it follows the same pattern as everything else. I didn't touch the startup code, the middleware, or any existing route.

I didn't change these two lines during the review round. What I *did* rely on is that the merged `ml_runtime` infrastructure already imports and registers its own router (`app.include_router(ml_runtime_router)`), which is what provides the authoritative `/ml/model-info`, `/ml/health`, `/ml/ready`, and `/ml/metrics`. So both routers sit under `/ml`, but they own **different** paths — no duplicates.

In [ ]:
# In the "Routers" import block:
from routers import ai_service as ai_router
from routers import ml_inference as ml_router     # <-- LINE 1 I ADDED
from routers import helpers as helpers_router
# ...
# (already present from the merged ml_runtime infra — NOT mine)
from ml_runtime import (
    MLRuntimeState,
    ModelMetadataError,
    capture_model_lineage,
    router as ml_runtime_router,
)

# In the section where routers are registered:
app.include_router(audiobooks_router.router)
app.include_router(ai_router.router)
app.include_router(ml_router.router)              # <-- LINE 2 I ADDED (my POST /ml/navigate)
app.include_router(helpers_router.router)
# ...
app.include_router(ml_runtime_router)             # already here — owns the authoritative /ml/model-info etc.


## 4. File I added: `tests/test_ml_inference.py` (router-isolated)

I wanted tests that prove the endpoint works **without downloading the real model weights** (weights aren't in git, and CI won't have them). So the trick I used is: I build a tiny throwaway FastAPI app in the test, mount *only* my router on it, and set `app.state.yolo` to a fake object that just has a `.names` dict. For the tests that need "inference", I `monkeypatch` `vision_adapter` to return a canned result. That way I'm testing my endpoint's logic and contract, not YOLO itself.

**What changed in review:** I deleted the two `GET /ml/model-info` tests (that endpoint isn't mine anymore), and my error tests now assert the **stable ML error shape** (`{"error": {"code": ..., "message": ...}}`) instead of the old `{"detail": ...}`. My canned result now uses `priority: "MEDIUM"` for `table` to match the contract, and the mock test asserts the approved 8 navigation classes.

Here's what each test checks now:

- **`test_navigate_contract_eight_class`** — happy path with an 8-name fake model. Checks the response has *exactly* the 7 contract keys, that `classes`/`model` came from the fake model's `.names` (so `walkbuddy-yolo-8class`), and that detections pass through unchanged.
- **`test_navigate_contract_seven_class`** — the *same* endpoint with a 7-name fake model. Proves I didn't hardcode the class list: `model` becomes `walkbuddy-yolo-7class`.
- **`test_navigate_empty_file_short_circuits`** — an empty upload returns a valid empty response (no crash, `image_id` is null).
- **`test_navigate_503_uses_stable_model_unavailable_error`** — if the model is missing, it returns 503 with the **stable** `model_unavailable` error body.
- **`test_navigate_500_uses_stable_inference_failed_error`** — if the adapter throws, it returns 500 with the **stable** `inference_failed` error body (temp file still cleaned up).
- **`test_navigate_mock_mode_reports_approved_navigation_classes`** — turns the env flag on with `yolo=None` and proves mock mode returns the deterministic fake in the exact same contract, reporting the approved 8 nav classes with `table` = `MEDIUM`, *with no weights loaded*.

The full current test file is below.

In [ ]:
"""
Router-isolated unit tests for POST /ml/navigate (routers/ml_inference.py).

These tests never load real model weights. They mount only the ml_inference
router on a bare FastAPI app, set a fake `app.state.yolo` (with a `.names`
mapping) plus a real capacity limiter, and stub out `vision_adapter` so the
contract can be verified without inference.

Note: `GET /ml/model-info` is intentionally NOT defined by this router anymore
(the authoritative one lives in ml_runtime). Errors use the stable ML error
format from ml_runtime.errors. Real-app wiring (single model-info, metrics,
route registration) is covered in tests/test_ml_inference_integration.py.

Run from the backend directory:
    pytest tests/test_ml_inference.py -v
"""

import anyio
import pytest
from fastapi import FastAPI
from fastapi.testclient import TestClient

import routers.ml_inference as ml_inference


SEVEN_CLASSES = {
    0: "book",
    1: "books",
    2: "monitor",
    3: "office-chair",
    4: "whiteboard",
    5: "table",
    6: "tv",
}

EIGHT_CLASSES = {**SEVEN_CLASSES, 7: "couch"}

# The approved MVP navigation taxonomy, derived from the shared contract.
APPROVED_NAV_CLASSES = ["person", "stairs", "door", "chair", "table", "pole", "bicycle", "vehicle"]


class FakeYolo:
    """Minimal stand-in for an Ultralytics YOLO model.

    Only `.names` is read by the endpoint; inference is stubbed separately.
    """

    def __init__(self, names):
        self.names = names


def _fake_result():
    return {
        "image_id": "frame",
        "detections": [
            {
                "category": "table",
                "confidence": 0.91,
                "bbox": {"x_min": 100, "y_min": 120, "x_max": 400, "y_max": 460},
                "direction": "ahead",
                "priority": "MEDIUM",
            }
        ],
        "metadata": {"image_shape": [480, 640]},
    }


def _build_app(yolo):
    app = FastAPI()
    app.include_router(ml_inference.router)
    app.state.yolo = yolo
    app.state.vision_limiter = anyio.CapacityLimiter(1)
    return app


@pytest.fixture
def stub_adapter(monkeypatch):
    """Replace vision_adapter with a canned result (no weights, no inference)."""
    monkeypatch.setattr(ml_inference, "vision_adapter", lambda model, path: _fake_result())


# ---------------------------------------------------------------------------
# /ml/navigate — contract (classes read from the model, not hardcoded)
# ---------------------------------------------------------------------------

def test_navigate_contract_eight_class(stub_adapter):
    client = TestClient(_build_app(FakeYolo(EIGHT_CLASSES)))
    resp = client.post(
        "/ml/navigate",
        files={"file": ("frame.jpg", b"not-a-real-jpeg", "image/jpeg")},
    )
    assert resp.status_code == 200
    body = resp.json()

    # Exact contract keys, nothing missing.
    assert set(body) == {
        "model",
        "classes",
        "detections",
        "guidance_message",
        "risk_level",
        "inference_time_ms",
        "image_id",
    }

    # classes + model are derived from app.state.yolo.names (not hardcoded).
    assert body["classes"] == [
        "book", "books", "monitor", "office-chair",
        "whiteboard", "table", "tv", "couch",
    ]
    assert body["model"] == "walkbuddy-yolo-8class"

    # Detections pass through verbatim from vision_adapter.
    assert body["detections"] == _fake_result()["detections"]
    assert body["image_id"] == "frame"
    assert isinstance(body["inference_time_ms"], int)
    assert isinstance(body["guidance_message"], str)
    assert isinstance(body["risk_level"], str) and body["risk_level"]


def test_navigate_contract_seven_class(stub_adapter):
    """Same endpoint must work unchanged for a different set of weights."""
    client = TestClient(_build_app(FakeYolo(SEVEN_CLASSES)))
    resp = client.post(
        "/ml/navigate",
        files={"file": ("frame.jpg", b"not-a-real-jpeg", "image/jpeg")},
    )
    assert resp.status_code == 200
    body = resp.json()
    assert body["model"] == "walkbuddy-yolo-7class"
    assert body["classes"] == [
        "book", "books", "monitor", "office-chair",
        "whiteboard", "table", "tv",
    ]
    assert "couch" not in body["classes"]


def test_navigate_empty_file_short_circuits(stub_adapter):
    client = TestClient(_build_app(FakeYolo(EIGHT_CLASSES)))
    resp = client.post(
        "/ml/navigate",
        files={"file": ("frame.jpg", b"", "image/jpeg")},
    )
    assert resp.status_code == 200
    body = resp.json()
    assert body["detections"] == []
    assert body["image_id"] is None
    assert body["model"] == "walkbuddy-yolo-8class"
    assert body["classes"][0] == "book"


# ---------------------------------------------------------------------------
# /ml/navigate — stable ML error format (not my own 503/500 shape)
# ---------------------------------------------------------------------------

def test_navigate_503_uses_stable_model_unavailable_error():
    client = TestClient(_build_app(yolo=None))
    resp = client.post(
        "/ml/navigate",
        files={"file": ("frame.jpg", b"not-a-real-jpeg", "image/jpeg")},
    )
    assert resp.status_code == 503
    assert resp.json() == {
        "error": {"code": "model_unavailable", "message": "Vision model is unavailable."}
    }


def test_navigate_500_uses_stable_inference_failed_error(monkeypatch):
    def _boom(model, path):
        raise RuntimeError("cuda exploded")

    monkeypatch.setattr(ml_inference, "vision_adapter", _boom)
    client = TestClient(_build_app(FakeYolo(EIGHT_CLASSES)))
    resp = client.post(
        "/ml/navigate",
        files={"file": ("frame.jpg", b"not-a-real-jpeg", "image/jpeg")},
    )
    assert resp.status_code == 500
    assert resp.json() == {
        "error": {"code": "inference_failed", "message": "Vision inference failed."}
    }


# ---------------------------------------------------------------------------
# Mock mode (WALKBUDDY_ML_MOCK) — approved navigation taxonomy, no weights
# ---------------------------------------------------------------------------

def test_navigate_mock_mode_reports_approved_navigation_classes(monkeypatch):
    monkeypatch.setenv("WALKBUDDY_ML_MOCK", "1")
    # yolo is None on purpose: mock mode must not require weights.
    client = TestClient(_build_app(yolo=None))
    resp = client.post(
        "/ml/navigate",
        files={"file": ("frame.jpg", b"ignored-in-mock", "image/jpeg")},
    )
    assert resp.status_code == 200
    body = resp.json()

    # Exact same contract as the real path.
    assert set(body) == {
        "model",
        "classes",
        "detections",
        "guidance_message",
        "risk_level",
        "inference_time_ms",
        "image_id",
    }

    # Approved MVP navigation taxonomy, in contract order.
    assert body["model"] == "walkbuddy-yolo-mock"
    assert body["classes"] == APPROVED_NAV_CLASSES

    # Priority comes from the contract's base severity (table is MEDIUM there).
    assert body["detections"] == [
        {
            "category": "table",
            "confidence": 0.87,
            "bbox": {"x_min": 220, "y_min": 180, "x_max": 420, "y_max": 400},
            "direction": "ahead",
            "priority": "MEDIUM",
        }
    ]
    assert body["image_id"] == "mock"
    assert body["inference_time_ms"] == 0
    assert isinstance(body["guidance_message"], str) and body["guidance_message"]
    assert isinstance(body["risk_level"], str) and body["risk_level"]


## 4b. File I added in review: `tests/test_ml_inference_integration.py`

The router-isolated tests above are great for checking my endpoint's logic in a vacuum, but the team lead pointed out they *can't* catch integration problems — like whether my route is actually registered on the real app, or whether I accidentally left a second `/ml/model-info` handler around. So in review I added a second test file that imports the **real** `main.app`.

The clever bit is that I import the actual app **but I never run its `lifespan`** (I use `TestClient(app)` without the `with` block). The lifespan is what loads the heavy stuff — YOLO weights, Whisper, the LLM — so skipping it means the tests are fast and need no weights. I just set by hand the two pieces of state that lifespan would normally create: a fresh `MLRuntimeState()` (so metric counts start clean each test) and a `CapacityLimiter`. I also stub `llama_cpp` with a `MagicMock` before importing `main`, because its native library isn't guaranteed to be installed in CI.

Here's what each integration test verifies, in plain words:

- **`test_navigate_registered_and_single_authoritative_model_info`** — `/ml/navigate` really is on the app, there's **exactly one** `/ml/model-info` route, and it belongs to `ml_runtime.router` (not me). It also asserts my module no longer even has a `model_info` function.
- **`test_existing_and_ml_endpoints_remain_intact`** — the old `/vision` and `/ws/vision` are still there, and so is the whole `/ml/*` runtime surface (`/ml/navigate`, `/ml/model-info`, `/ml/health`, `/ml/ready`, `/ml/metrics`). This is my "pure addition" guarantee.
- **`test_mock_mode_reports_approved_navigation_classes`** — through the real app, mock mode returns the approved 8 navigation classes with `table` = `MEDIUM`.
- **`test_metrics_update_on_successful_inference`** — after a successful call, the shared metrics show `total_attempts +1` and `successful_inferences +1`.
- **`test_metrics_update_on_failed_inference`** — when the adapter blows up, I get the stable `inference_failed` 500 body **and** the metrics show `total_attempts +1` and `failed_inferences +1`.
- **`test_model_unavailable_uses_stable_error`** — no model loaded → stable `model_unavailable` 503 body.

The full integration test file is below.

In [ ]:
"""
Real-app integration tests for the ML inference wiring.

Unlike test_ml_inference.py (which mounts only my router in isolation), these
tests import the actual `main.app` — with every router, middleware, and the
shared `ml_runtime` state wired exactly as production does — and verify the
integration the team lead asked for:

  * POST /ml/navigate is registered on the real app.
  * There is exactly ONE authoritative GET /ml/model-info, and it belongs to
    ml_runtime (my duplicate handler has been removed).
  * Mock mode reports the approved 8 navigation classes.
  * Shared ML-runtime metrics update on BOTH successful and failed inference.
  * Existing /vision, /ws/vision and the /ml/* runtime endpoints stay intact.

The app's lifespan is intentionally NOT run (TestClient is used without its
context manager), so no real model weights, OCR, Whisper, or LLM are loaded.
We set the two pieces of state that lifespan would normally create.

Run from the backend directory:
    pytest tests/test_ml_inference_integration.py -v
"""

import sys
from unittest.mock import MagicMock

import anyio
import pytest
from fastapi.testclient import TestClient

# llama_cpp's native library may be absent (it's not a hard test dependency).
# Stub it before importing main so `from slow_lane import SlowLaneBrain` works.
if "llama_cpp" not in sys.modules:
    sys.modules["llama_cpp"] = MagicMock()

import main  # noqa: E402  (import after the llama_cpp guard above)
import routers.ml_inference as ml_inference  # noqa: E402
from ml_runtime import MLRuntimeState  # noqa: E402


APPROVED_NAV_CLASSES = ["person", "stairs", "door", "chair", "table", "pole", "bicycle", "vehicle"]


class FakeYolo:
    def __init__(self, names):
        self.names = names


def _fake_result():
    return {
        "image_id": "frame",
        "detections": [
            {
                "category": "table",
                "confidence": 0.91,
                "bbox": {"x_min": 100, "y_min": 120, "x_max": 400, "y_max": 460},
                "direction": "ahead",
                "priority": "MEDIUM",
            }
        ],
        "metadata": {"image_shape": [480, 640]},
    }


@pytest.fixture
def app():
    """The real production app, with the state lifespan would create.

    A fresh MLRuntimeState per test keeps metric deltas isolated. The lifespan
    is not executed, so no heavy models load.
    """
    application = main.app
    application.state.ml_runtime = MLRuntimeState()
    application.state.vision_limiter = anyio.CapacityLimiter(1)
    application.state.yolo = None
    return application


def _routes_with_path(application, path):
    return [r for r in application.routes if getattr(r, "path", None) == path]


# ---------------------------------------------------------------------------
# Route registration + single authoritative /ml/model-info
# ---------------------------------------------------------------------------

def test_navigate_registered_and_single_authoritative_model_info(app):
    paths = {getattr(r, "path", None) for r in app.routes}
    assert "/ml/navigate" in paths

    info_routes = _routes_with_path(app, "/ml/model-info")
    # Exactly one handler for the shared path...
    assert len(info_routes) == 1
    # ...and it is ml_runtime's, not the one I used to define in ml_inference.
    assert info_routes[0].endpoint.__module__ == "ml_runtime.router"
    assert not hasattr(ml_inference, "model_info")


def test_existing_and_ml_endpoints_remain_intact(app):
    paths = {getattr(r, "path", None) for r in app.routes}
    # Existing vision routes untouched.
    assert "/vision" in paths
    assert "/ws/vision" in paths
    # My endpoint plus the full ml_runtime surface.
    for expected in ["/ml/navigate", "/ml/model-info", "/ml/health", "/ml/ready", "/ml/metrics"]:
        assert expected in paths, expected


# ---------------------------------------------------------------------------
# Mock mode via the real app
# ---------------------------------------------------------------------------

def test_mock_mode_reports_approved_navigation_classes(app, monkeypatch):
    monkeypatch.setenv("WALKBUDDY_ML_MOCK", "1")
    client = TestClient(app)  # no context manager -> lifespan is not run
    resp = client.post("/ml/navigate", files={"file": ("f.jpg", b"x", "image/jpeg")})
    assert resp.status_code == 200
    body = resp.json()
    assert body["model"] == "walkbuddy-yolo-mock"
    assert body["classes"] == APPROVED_NAV_CLASSES
    assert body["detections"][0]["priority"] == "MEDIUM"  # table is MEDIUM in the contract


# ---------------------------------------------------------------------------
# Shared ML-runtime metrics update on success AND failure
# ---------------------------------------------------------------------------

def test_metrics_update_on_successful_inference(app, monkeypatch):
    monkeypatch.delenv("WALKBUDDY_ML_MOCK", raising=False)
    monkeypatch.setattr(ml_inference, "vision_adapter", lambda model, path: _fake_result())
    app.state.yolo = FakeYolo({0: "table"})
    client = TestClient(app)

    before = app.state.ml_runtime.metrics.snapshot()
    resp = client.post("/ml/navigate", files={"file": ("f.jpg", b"x", "image/jpeg")})
    assert resp.status_code == 200
    after = app.state.ml_runtime.metrics.snapshot()

    assert after["total_attempts"] == before["total_attempts"] + 1
    assert after["successful_inferences"] == before["successful_inferences"] + 1
    assert after["failed_inferences"] == before["failed_inferences"]
    assert after["last_inference_at"] is not None


def test_metrics_update_on_failed_inference(app, monkeypatch):
    monkeypatch.delenv("WALKBUDDY_ML_MOCK", raising=False)

    def _boom(model, path):
        raise RuntimeError("boom")

    monkeypatch.setattr(ml_inference, "vision_adapter", _boom)
    app.state.yolo = FakeYolo({0: "table"})
    client = TestClient(app)

    before = app.state.ml_runtime.metrics.snapshot()
    resp = client.post("/ml/navigate", files={"file": ("f.jpg", b"x", "image/jpeg")})
    assert resp.status_code == 500
    assert resp.json() == {
        "error": {"code": "inference_failed", "message": "Vision inference failed."}
    }
    after = app.state.ml_runtime.metrics.snapshot()

    assert after["total_attempts"] == before["total_attempts"] + 1
    assert after["failed_inferences"] == before["failed_inferences"] + 1
    assert after["successful_inferences"] == before["successful_inferences"]


def test_model_unavailable_uses_stable_error(app, monkeypatch):
    monkeypatch.delenv("WALKBUDDY_ML_MOCK", raising=False)
    app.state.yolo = None
    client = TestClient(app)
    resp = client.post("/ml/navigate", files={"file": ("f.jpg", b"x", "image/jpeg")})
    assert resp.status_code == 503
    assert resp.json() == {
        "error": {"code": "model_unavailable", "message": "Vision model is unavailable."}
    }

## 5. File I added: `requirements-dev.txt`

When I first tried to run my tests, I found `pytest` wasn't actually installed in the project's virtual environment, even though the existing tests assume it. That means CI wouldn't be able to run them either. So I added a `requirements-dev.txt` for test-only dependencies.

I kept it simple: it uses `-r requirements.txt` to pull in all the normal runtime deps (which already include `httpx`, the thing FastAPI's `TestClient` needs), and then adds `pytest` pinned to the version I verified with. This matches the repo's "one pinned requirements file" style but keeps test-only stuff separate from what the server needs to run in production.

In [ ]:
# requirements-dev.txt

# Test-only dependencies (not needed at runtime).
# Install with:  pip install -r requirements-dev.txt
#
# Runtime deps (including httpx, which FastAPI's TestClient uses) live in
# requirements.txt and are pulled in via the -r line below.
-r requirements.txt

pytest==9.1.1


## 6. File I added: `docs/ml_inference.md` (summary)

I wrote a markdown doc so anyone (frontend devs, reviewers) can understand and use the endpoint without reading the code. I updated it in review too. In my own words, it now covers:

- **What it is** — a pure addition that serves the model through the existing pipeline, doesn't change any existing route. It's scoped to just `POST /ml/navigate`, and notes that `GET /ml/model-info` is owned by `ml_runtime.router`.
- **Design** — how it reuses `app.state.yolo`, the limiter, `vision_adapter`, and the guidance helpers; how `model`/`classes` come from `yolo.names`; and that it participates in the shared ML-runtime **metrics** and uses the **stable error format**.
- **Mock mode** — the `WALKBUDDY_ML_MOCK` flag, off by default, works with no weights, returns `walkbuddy-yolo-mock`, and reports the approved navigation taxonomy + priorities **derived from `ml_contract/navigation_semantics.py`**.
- **The endpoint** — `POST /ml/navigate` with an example JSON response (mock example shows `table` at `MEDIUM`).
- **Error handling** — the stable 503 (`model_unavailable`) and 500 (`inference_failed`) bodies from `ml_runtime.errors`, and temp-file cleanup.
- **Metrics** — how each real inference call updates attempts / successes / failures / latency, visible at `GET /ml/metrics`.
- **The class-lineage note** — points at both the weights/config discrepancy and `ml_contract/navigation_semantics.py` as the source of the approved taxonomy.
- **How to run the tests** — install `requirements-dev.txt`, then `pytest` on both test files.

The shell cell below shows the quick-start bit from that doc (how to flip on mock mode).

In [ ]:
# Mock mode: accepts 1/true/yes/on; unset (or anything else) = off
export WALKBUDDY_ML_MOCK=1

# Now /ml/navigate returns a deterministic fake result in the exact same
# contract, with NO weights loaded and NO inference. The "model" field is
# "walkbuddy-yolo-mock" so the frontend knows it's fake, and the classes are
# the approved navigation taxonomy derived from ml_contract.
# (GET /ml/model-info is served by ml_runtime, not by my router.)


## 7. How I tested it

I tested two ways: automated tests (the important one for CI) and manual `curl` commands against a running server.

**Automated:** I now have **12 targeted tests** across the two files — 6 router-isolated (`test_ml_inference.py`) + 6 real-app integration (`test_ml_inference_integration.py`). They all pass and, crucially, none of them need the real `best.pt` weights — they use the fake model + monkeypatched adapter (and for integration, they skip the app's `lifespan` so no heavy models load).

I also ran the **full backend suite** to make sure my change didn't break anything else: **127 passed, 1 failed**. The single failure is a *pre-existing, unrelated* one — `predictive_path/test_predictive_path.py::test_full_scenarios` fails with an `ImportError` (it imports `PredictivePathSystem`, a name that doesn't exist in that package). It's in a different feature area, it fails the same way on a clean checkout, and none of my files touch it. All of the vision/ML tests pass.

> Note: my earlier PR description said "121 passed" — that number is now stale because more tests were added to the repo since I opened the PR. The real current numbers are the ones above.

**Manual:** if the server is running (`python main.py`), I can hit the endpoint with `curl`. In mock mode I don't even need weights. (If `WALKBUDDY_API_KEY` is set on the server, add `-H "X-API-Key: $WALKBUDDY_API_KEY"`.)

The commands I used are in the shell cell below, and the real pytest output is in the cell after it.

In [ ]:
# --- Automated tests (from the backend directory) ---
pip install -r requirements-dev.txt

# My 12 targeted tests (router-isolated + real-app integration)
pytest tests/test_ml_inference.py tests/test_ml_inference_integration.py -v

# The full backend suite (to prove I didn't break anything)
pytest

# --- Manual smoke test with curl (server must be running) ---

# Ask which classes the model knows (served by ml_runtime, not my router)
curl http://localhost:8000/ml/model-info

# Send a frame and get detections + guidance
curl -X POST http://localhost:8000/ml/navigate \
  -F "file=@frame.jpg"

# Same, but in mock mode (no weights required) — start the server with the flag:
#   WALKBUDDY_ML_MOCK=1 python main.py
curl -X POST http://localhost:8000/ml/navigate -F "file=@frame.jpg"
# -> {"model":"walkbuddy-yolo-mock", "classes":["person","stairs","door","chair","table","pole","bicycle","vehicle"],
#     "detections":[{"category":"table",...,"priority":"MEDIUM"}], ...}


This is the actual output I got — all **12 targeted tests** passing:

```
============================= test session starts ==============================
platform darwin -- Python 3.11.15, pytest-9.1.1, pluggy-1.6.0 -- .../.venv/bin/python
cachedir: .pytest_cache
rootdir: .../software_side/walkbuddy_reactNative/backend
plugins: anyio-4.12.1
collecting ... collected 12 items

tests/test_ml_inference.py::test_navigate_contract_eight_class PASSED    [  8%]
tests/test_ml_inference.py::test_navigate_contract_seven_class PASSED    [ 16%]
tests/test_ml_inference.py::test_navigate_empty_file_short_circuits PASSED [ 25%]
tests/test_ml_inference.py::test_navigate_503_uses_stable_model_unavailable_error PASSED [ 33%]
tests/test_ml_inference.py::test_navigate_500_uses_stable_inference_failed_error PASSED [ 41%]
tests/test_ml_inference.py::test_navigate_mock_mode_reports_approved_navigation_classes PASSED [ 50%]
tests/test_ml_inference_integration.py::test_navigate_registered_and_single_authoritative_model_info PASSED [ 58%]
tests/test_ml_inference_integration.py::test_existing_and_ml_endpoints_remain_intact PASSED [ 66%]
tests/test_ml_inference_integration.py::test_mock_mode_reports_approved_navigation_classes PASSED [ 75%]
tests/test_ml_inference_integration.py::test_metrics_update_on_successful_inference PASSED [ 83%]
tests/test_ml_inference_integration.py::test_metrics_update_on_failed_inference PASSED [ 91%]
tests/test_ml_inference_integration.py::test_model_unavailable_uses_stable_error PASSED [100%]

============================== 12 passed in 5.15s ==============================
```

And the **full backend suite** — 127 passed, with the single pre-existing, unrelated `predictive_path` failure I mentioned above:

```
=========================== short test summary info ============================
FAILED predictive_path/test_predictive_path.py::test_full_scenarios - ImportError: cannot import name 'PredictivePathSystem' from 'predictive_path'
================== 1 failed, 127 passed, 6 warnings in 13.54s ==================
```

## 8. What I learned / next steps

**What I learned:**

- **Reuse beats rewriting.** The biggest win was realising the model, limiter, adapter, and guidance helpers already existed. My job was really just to wire them into a clean new contract, not to write inference code. Reading the existing code first saved me a lot of work and kept behaviour consistent.
- **Don't hardcode data that the model already carries.** Reading class names from `yolo.names` at runtime is what makes the endpoint survive the 7-vs-8 class confusion. If I'd typed the class list into my code, it would silently be wrong for one of the two models.
- **Mock mode is genuinely useful.** Being able to develop the API and frontend before real weights exist (and to run tests with no weights) removes a big blocker for the team.
- **Match the existing error style.** Reusing the stable ML error format and the temp-file `finally` cleanup means my endpoint behaves like the rest of the backend, so there are no surprises.

**What the review taught me (this is the big one):**

- **Rebase onto the latest branch *before* integrating.** While my PR was open, a lot of new ML infrastructure (`ml_contract`, `ml_runtime`) landed on `t2-2026-development`. If I'd kept building on my old base I'd have collided with it. Pulling the latest first and rebasing meant I was integrating with what the team actually has, not a stale snapshot.
- **Don't rebuild what the team already built.** I had my own `GET /ml/model-info`, but `ml_runtime` now provides the authoritative one. Two handlers on the same path is confusing and fragile (which one wins depends on registration order), so the right move was to *delete mine* and defer to the shared one. Same idea with metrics and error formats — I plugged into the shared helpers instead of inventing my own 503/500 shapes.
- **Derive from a central contract instead of hardcoding.** The approved taxonomy and per-class severities live in `ml_contract/navigation_semantics.py`. Deriving `MOCK_CLASSES` and the mock priority from there means there's a single source of truth, so if the team changes the taxonomy my endpoint follows automatically instead of drifting out of sync.

**Next steps / follow-ups (out of scope for this task):**

- **Resolve the 7-vs-8 class lineage mismatch.** The config says 8 classes (`couch`) but the verified `best.pt` has 7. Someone needs to confirm which model is the real production one and line up the config/weights. My endpoint already reports whatever is loaded, so it won't break either way — but the underlying mismatch should still be fixed.
- **Real latency numbers.** Right now `inference_time_ms` is measured but I haven't benchmarked it against actual weights on the target hardware.
- **Maybe add distance estimation.** The contract has room for it (the guidance events carry `distance_m`), but that's a separate feature.
- **Consider adding `pytest` to CI config** so these tests actually run on every PR now that `requirements-dev.txt` exists.